<a href="https://colab.research.google.com/github/BraedynL0530/Aenaos/blob/main/PaperProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from transformers import OwlViTForObjectDetection, OwlViTProcessor
from PIL import Image

processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch16")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch16")

model.eval()
model.config.output_hidden_states = True


# ----------------------------
# Utils
# ----------------------------

def box_area(box):
    return box[2] * box[3]


def iou(a, b):
    def xyxy(box):
        cx, cy, w, h = box
        return (
            cx - w / 2,
            cy - h / 2,
            cx + w / 2,
            cy + h / 2,
        )

    a1 = xyxy(a)
    a2 = xyxy(b)

    xi1 = max(a1[0], a2[0])
    yi1 = max(a1[1], a2[1])
    xi2 = min(a1[2], a2[2])
    yi2 = min(a1[3], a2[3])

    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    area1 = (a1[2] - a1[0]) * (a1[3] - a1[1])
    area2 = (a2[2] - a2[0]) * (a2[3] - a2[1])

    return inter / (area1 + area2 - inter + 1e-6)


# ----------------------------
# NMS-style dedup
# ----------------------------

def dedup(boxes, scores, iou_thresh=0.4):
    keep = []

    idxs = scores.argsort(descending=True)

    for i in idxs:
        box = boxes[i]

        if box_area(box) < 0.005:
            continue

        if all(iou(box, k) < iou_thresh for k in keep):
            keep.append(box)

    return torch.stack(keep) if len(keep) > 0 else torch.empty((0, 4))


# ----------------------------
# Inference
# ----------------------------

image = Image.open("/content/Untitled.jpg")

text_queries = [
    "dog",
    "animal",
    "hand",
    "person",
    "object",
    "foreground object",
]

inputs = processor(images=image, text=text_queries, return_tensors="pt")
outputs = model(**inputs)

boxes = outputs.pred_boxes[0]   # (N, 4)
logits = outputs.logits[0]      # (N, num_queries)

# ----------------------------
# BETTER scoring (ranking not thresholding)
# ----------------------------

scores = logits.max(dim=-1).values  # raw similarity

# take top-k instead of threshold
TOP_K = min(30, scores.shape[0])
scores, idx = scores.topk(TOP_K)

boxes = boxes[idx]

# ----------------------------
# cleanup
# ----------------------------

clean_boxes = dedup(boxes, scores)

# ----------------------------
# world-model-ready output
# ----------------------------

objects = [
    {
        "bbox": b.tolist(),   # cx,cy,w,h
        "state": "unknown"
    }
    for b in clean_boxes
]

print(f"Final objects: {len(objects)}")

for o in objects:
    print(o)

In [3]:
import torch
import torch.nn as nn

class worldGroundedEncoder(nn.Module):#removed delta and iou for attention based implict
  def __init__(self,spatial_dim=128,owl_dim=768,output_dim=256):
    super().__init__()
    self.delta_threshold = delta_threshold

    self.spatial_embd = nn.Sequential(
        nn.Linear(4,spatial_dim),
        nn.ReLU(),
        nn.LayerNorm(spatial_dim),
        nn.Linear(spatial_dim,output_dim),
        nn.LayerNorm(output_dim)
    )
    self.features = nn.Sequential(
        nn.Linear(owl_dim,output_dim), # may add a linear inbetween
        nn.ReLU(),
        nn.LayerNorm(output_dim)
    )
    # The input dimension for final_features must be 2 * output_dim after concatenation
    self.final_features = nn.Linear(2 * output_dim, output_dim) # Fixed input_dim based on concatenation


  def forward(self, norm_bbox,norm_quries):
    spatial = self.spatial_embd(norm_bbox)
    visual_features = self.features(norm_quries)

    enriched_features = torch.cat((spatial,visual_features),dim=1)
    final_features = self.final_features(enriched_features)
    return final_features


In [4]:
class DynamicsPredictor(nn.Module):
  def __init__(self,feature_dim=256,layers=1):
    super().__init__()
    self.temporal_mem = nn.LSTM(
        input_size=feature_dim,
        hidden_size=feature_dim,
        num_layers=layers,
        batch_first=True
    )

    self.interaction = nn.MultiHeadAttention(embed_dim=feature_dim,num_heads=8,batch_first=True)

    self.pred_features = nn.Linear(feature_dim, feature_dim)
    self.pred_bbox = nn.Linear(feature_dim, 4)

  def forward(self, current_feats, prev_state=None):
        """
        current_feats: (num_objects, 1, feature_dim)
        """
        # Pass through temporal tracking
        lstm_out, new_state = self.lstm(current_feats, prev_state) # (num_objects, 1, feature_dim)

        # Collapse sequence dim to model object interactions across space
        scene_space = lstm_out.squeeze(1).unsqueeze(0) #squeeze acts like remove unsqueeze acts like insert for tensor

        attn_out, _ = self.interaction(scene_space, scene_space, scene_space)
        attn_out = attn_out.squeeze(0) # (num_objects, feature_dim)

        # Predict physical state at t+1
        next_feats = self.pred_features(attn_out)
        next_boxes = torch.sigmoid(self.pred_bbox(attn_out)) # Keep boxes normalized in [0, 1]

        return next_feats, next_boxes, new_state

In [ ]:
class CounterfactualSimulator(nn.Module):
  pass

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/video_reasoning_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# ---------- Dataset (dummy example – replace with CLEVRER loader) ----------
class TrackDataset(torch.utils.data.Dataset):
    def __init__(self, tracks, wge):
        # tracks: list of lists of (bbox, query) per object
        self.tracks = tracks
        self.wge = wge

    def __len__(self): return len(self.tracks)

    def __getitem__(self, idx):
        track = self.tracks[idx]  # list of (bbox, query) for T frames
        T = len(track) - 1
        bbox_seq, query_seq, target_bbox, target_feat = [], [], [], []
        for t in range(T):
            bbox_seq.append(track[t][0])
            query_seq.append(track[t][1])
            # target: next frame's bbox and feature (precomputed with frozen wge)
            target_bbox.append(track[t+1][0])
            with torch.no_grad():
                target_feat.append(self.wge(track[t+1][0].unsqueeze(0), track[t+1][1].unsqueeze(0))[0])
        return (torch.stack(bbox_seq), torch.stack(query_seq),
                torch.stack(target_bbox), torch.stack(target_feat))

# ---------- Training ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
wge = worldGroundedEncoder().to(device)
dyn_pred = DynamicsPredictor().to(device)
optimizer = torch.optim.AdamW(list(wge.parameters())+list(dyn_pred.parameters()), lr=1e-4)
loss_bbox = nn.L1Loss()
loss_feat = nn.CosineEmbeddingLoss()  # (x1, x2, target=1) => minimises 1-cosine

# Assume loader returns (B,T,4), (B,T,768), (B,T,4), (B,T,256)
# loader = DataLoader(TrackDataset(all_tracks, wge), batch_size=32, shuffle=True)

for epoch in range(20):
    wge.train(); dyn_pred.train()
    epoch_loss = 0
    for bbox_seq, query_seq, tgt_bbox, tgt_feat in loader:  # shapes: (B,T,…)
        bbox_seq, query_seq = bbox_seq.to(device), query_seq.to(device)
        tgt_bbox, tgt_feat = tgt_bbox.to(device), tgt_feat.to(device)
        B, T, _ = bbox_seq.shape
        total_loss = 0
        prev_state = None
        for t in range(T):
            curr_feat = wge(bbox_seq[:,t], query_seq[:,t]).unsqueeze(1)  # (B,1,256)
            pred_feat, pred_box, new_state = dyn_pred(curr_feat, prev_state)
            loss_b = loss_bbox(pred_box, tgt_bbox[:,t])
            loss_f = loss_feat(pred_feat, tgt_feat[:,t], torch.ones(B).to(device))
            total_loss += (loss_b + loss_f)
            prev_state = new_state
        total_loss /= T
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        epoch_loss += total_loss.item()
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(loader):.4f}")
    # Checkpoint every 5 epochs
    if (epoch+1) % 5 == 0:
        torch.save({'epoch':epoch, 'wge':wge.state_dict(), 'dyn':dyn_pred.state_dict(),
                    'opt':optimizer.state_dict()}, f"{CKPT_DIR}/ckpt_{epoch+1}.pt")

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
import numpy as np
from PIL import Image
import torch, torch.nn.functional as F

# ---------- Load models ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
wge = worldGroundedEncoder().to(device)
dyn_pred = DynamicsPredictor().to(device)
# Load checkpoint if available:
# ckpt = torch.load('latest.pt', map_location=device)
# wge.load_state_dict(ckpt['wge']); dyn_pred.load_state_dict(ckpt['dyn'])
wge.eval(); dyn_pred.eval()

# Owl-ViT (from transformers)
from transformers import OwlViTForObjectDetection, OwlViTProcessor
owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch16").to(device)
owl_processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch16")
owl_model.config.output_hidden_states = True

# ---------- Simple IoU matching ----------
def box_iou(b1, b2):
    def to_xyxy(b): return (b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2)
    x1, y1, x2, y2 = to_xyxy(b1); x1g, y1g, x2g, y2g = to_xyxy(b2)
    inter = (min(x2, x2g)-max(x1, x1g)).clamp(0) * (min(y2, y2g)-max(y1, y1g)).clamp(0)
    area1 = (x2-x1)*(y2-y1); area2 = (x2g-x1g)*(y2g-y1g)
    return inter / (area1 + area2 - inter + 1e-6)

def match(cache, det_boxes, iou_th=0.3):
    matched, used = {}, set()
    for oid, v in cache.items():
        ious = [box_iou(v['bbox'], db).item() for db in det_boxes]
        best = np.argmax(ious)
        if ious[best] >= iou_th and best not in used:
            matched[oid] = best; used.add(best)
    return matched, [i for i in range(len(det_boxes)) if i not in used]

# ---------- Inference on video frames ----------
def run_inference(video_frames, delta_th=0.85):
    cache = {}          # id -> {bbox, feat, lstm_state, pred_next_feat}
    id_counter = 0
    stream = []         # surprise events for NLP
    frame_results = []

    for f_idx, image in enumerate(video_frames):
        inputs = owl_processor(images=image, text=["object"], return_tensors="pt").to(device)
        out = owl_model(**inputs)
        boxes = out.pred_boxes[0]                     # (Q,4) cxcywh
        queries = out.decoder_hidden_states[-1][0]    # (Q,768)
        scores = torch.sigmoid(out.logits.max(-1).values)[0]
        valid = scores > 0.1
        dboxes, dqueries = boxes[valid], queries[valid]

        matched, unmatched = match(cache, dboxes)
        # add new objects
        for idx in unmatched:
            id_counter += 1
            oid = id_counter
            feat = wge(dboxes[idx].unsqueeze(0), dqueries[idx].unsqueeze(0))[0]
            cache[oid] = {'bbox': dboxes[idx], 'feat': feat, 'lstm_state': None, 'pred_feat': None}

        active = list(matched.keys()) + list(range(id_counter-len(unmatched)+1, id_counter+1))
        # Process each active object sequentially (LSTM + interaction can be done in mini-batch)
        for oid in active:
            det_idx = matched.get(oid, unmatched[active.index(oid)])  # map id to detection index
            curr_feat = wge(dboxes[det_idx].unsqueeze(0), dqueries[det_idx].unsqueeze(0)).unsqueeze(1)  # (1,1,256)
            prev_state = cache[oid].get('lstm_state')
            pred_feat, pred_box, new_state = dyn_pred(curr_feat, prev_state)
            cache[oid]['lstm_state'] = new_state

            # Delta check
            old_pred = cache[oid].get('pred_feat')
            if old_pred is not None:
                sim = F.cosine_similarity(old_pred.unsqueeze(0), curr_feat.squeeze(0), dim=-1).item()
                if sim < delta_th:
                    stream.append(f"Frame {f_idx}: Entity {oid} deviated from prediction.")
                    # You can append more detailed features later
            cache[oid]['pred_feat'] = pred_feat[0]
            cache[oid]['bbox'] = dboxes[det_idx]
            cache[oid]['feat'] = curr_feat[0,0]
            cache[oid]['pred_box'] = pred_box[0]

        # Visualization (every 10 frames or at the end)
        if f_idx % 10 == 0:
            fig, ax = plt.subplots(figsize=(8,6))
            ax.imshow(image)
            for oid in active:
                bbox = cache[oid]['bbox'].cpu().numpy()  # cxcywh
                x, y, w, h = bbox[0]-bbox[2]/2, bbox[1]-bbox[3]/2, bbox[2], bbox[3]
                x, y, w, h = x*image.width, y*image.height, w*image.width, h*image.height
                color = 'red' if oid in stream else 'lime'
                ax.add_patch(patches.Rectangle((x,y), w, h, fill=False, edgecolor=color, lw=2))
                ax.text(x, y-5, str(oid), color='white', fontsize=8,
                        bbox=dict(facecolor=color, alpha=0.5))
                if 'pred_box' in cache[oid]:
                    pb = cache[oid]['pred_box'].cpu().numpy()
                    px, py, pw, ph = pb[0]-pb[2]/2, pb[1]-pb[3]/2, pb[2], pb[3]
                    px, py = px*image.width, py*image.height
                    pw, ph = pw*image.width, ph*image.height
                    ax.add_patch(patches.Rectangle((px,py), pw, ph, fill=False, edgecolor='yellow', lw=1, linestyle='--'))
            plt.title(f"Frame {f_idx}"); plt.axis('off'); plt.show()

    return stream, cache

# ---------- Q&A with Phi-3-mini ----------
def ask_brain(question, stream, model_name="microsoft/Phi-3-mini-4k-instruct"):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).eval()
    events = '\n'.join(stream[-30:])
    prompt = f"<|user|>\nVideo events:\n{events}\n\nQuestion: {question}\n<|assistant|>\n"
    inputs = tok(prompt, return_tensors='pt')
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7)
    return tok.decode(out[0], skip_special_tokens=True).split('<|assistant|>')[-1]

# Example usage:
# stream, _ = run_inference(video_frames)
# print(ask_brain("What caused the cup to fall?", stream))